In [ ]:
import os
import sys
import pickle
import time
import lmdb
import numpy as np
import torch
import matplotlib.pyplot as plt
from pathlib import Path

# Setup paths and environment
project_root = Path("..").resolve()
src_dir = project_root / "src"
sys.path.insert(0, str(src_dir))

# Configure OCIO for PyOpenColorIO
ocio_config_path = project_root / "config" / "aces" / "studio-config.ocio"
os.environ["OCIO"] = str(ocio_config_path)

import PyOpenColorIO as ocio

# Import our custom modules
from luminascale.utils.dataset_pair_generator import DatasetPairGenerator

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Open LMDB
lmdb_path = project_root / "dataset" / "training_data.lmdb"
if 'env' in locals() and env is not None:
    env.close()

env = lmdb.open(str(lmdb_path), readonly=True, lock=False, max_readers=1024)
print(f"LMDB opened: {lmdb_path}")


In [ ]:
# Initialize pair generators with mode parameter
bde_gen = DatasetPairGenerator(env, device, mode="bde")
cc_gen = DatasetPairGenerator(env, device, mode="cc")

# Cache keys for efficient iteration
keys_list = bde_gen.keys_cache
print(f"Dataset has {len(keys_list)} images")


In [ ]:
import time
import matplotlib.pyplot as plt

print("--- Test 1: BDE Single Image (GPU Pipeline) ---")

# Use generator to get a single pair (reference 32-bit, synthetic 8-bit)
bde_gen_iter = bde_gen.generate_pairs(num_images=1, start_idx=69)
ref_32f, syn_8u, look_params = next(bde_gen_iter)

print(f"Reference (32-bit sRGB) shape: {ref_32f.shape}, dtype: {ref_32f.dtype}")
print(f"  Stats: min={ref_32f.min():.4f}, max={ref_32f.max():.4f}, mean={ref_32f.mean():.4f}")
print(f"Synthetic (8-bit sRGB) shape: {syn_8u.shape}, dtype: {syn_8u.dtype}")
print(f"  Range: [0, 255]")

# Visualize
ref_32bit_np = ref_32f.cpu().numpy()
syn_8bit_np = syn_8u.cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(15, 7))
axes[0].imshow(np.clip(ref_32bit_np, 0, 1))
axes[0].set_title("BDE Reference: 32-bit sRGB (Truth)")
axes[1].imshow(syn_8bit_np)
axes[1].set_title("BDE Synthetic: 8-bit sRGB (Input)")
plt.tight_layout()
plt.show()


In [ ]:
import time
import matplotlib.pyplot as plt

print("--- Test 2: Color Convert Single Image (GPU Pipeline) ---")

# Use generator to get a single pair (reference ACES, synthetic sRGB)
cc_gen_iter = cc_gen.generate_pairs(num_images=1, start_idx=1)
ref_aces, syn_srgb, look_params_cc = next(cc_gen_iter)

print(f"Reference (ACES 32-bit) shape: {ref_aces.shape}, dtype: {ref_aces.dtype}")
print(f"  Stats: min={ref_aces.min():.4f}, max={ref_aces.max():.4f}")
print(f"Synthetic (sRGB 32-bit) shape: {syn_srgb.shape}, dtype: {syn_srgb.dtype}")
print(f"  Stats: min={syn_srgb.min():.4f}, max={syn_srgb.max():.4f}, mean={syn_srgb.mean():.4f}")

# Visualize sRGB (ACES is linear, so harder to visualize directly)
syn_srgb_np = syn_srgb.cpu().numpy()

plt.figure(figsize=(10, 8))
plt.imshow(np.clip(syn_srgb_np, 0, 1))
plt.title("Color Convert: Synthetic sRGB (Input to Convert to ACES)")
plt.tight_layout()
plt.show()


In [ ]:
from tqdm import tqdm

def run_bde_pipeline_test(num_imgs=20):
    print(f"--- Test 3: BDE Pipeline ({num_imgs} images) ---")
    
    # Warm up shaders with dummy tensor
    H_ref, W_ref = 3670, 5496
    aces_warm = torch.zeros((H_ref, W_ref, 3), device=device)
    _ = bde_gen.ocio_processor.apply_ocio_torch(aces_warm)
    
    t_start = time.time()
    count = 0
    for ref, syn, look in bde_gen.generate_pairs(num_images=num_imgs):
        count += 1
    t_end = time.time()
    
    total_time = t_end - t_start
    avg_ms = (total_time / count) * 1000 if count > 0 else 0
    print(f"Total Time ({count} images): {total_time:.2f} s")
    print(f"Average Time per image: {avg_ms:.2f} ms")
    return avg_ms

_ = run_bde_pipeline_test(20)


In [ ]:
from tqdm import tqdm

def run_cc_pipeline_test(num_imgs=20):
    print(f"--- Test 4: Color Convert Pipeline ({num_imgs} images) ---")
    
    # Warm up shaders with dummy tensor
    H_ref, W_ref = 3670, 5496
    aces_warm = torch.zeros((H_ref, W_ref, 3), device=device)
    _ = cc_gen.ocio_processor.apply_ocio_torch(aces_warm)

    t_start = time.time()
    count = 0
    for ref, syn, look in cc_gen.generate_pairs(num_images=num_imgs):
        count += 1
    t_end = time.time()
    
    total_time = t_end - t_start
    avg_ms = (total_time / count) * 1000 if count > 0 else 0
    print(f"Total Time ({count} images): {total_time:.2f} s")
    print(f"Average Time per image: {avg_ms:.2f} ms")
    return avg_ms

_ = run_cc_pipeline_test(20)


In [ ]:
# Cleanup GPU resources
bde_gen.cleanup()
cc_gen.cleanup()
env.close()
print("✅ Cleanup complete")
